In [ ]:
import pandas as pd 
import warnings

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('data/training_data.csv')

In [ ]:
df.head()

In [ ]:
df = df[df['class'] != 5]

## Eda

In [ ]:
df.tail()

In [ ]:
df[df['class'] == 1]

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.query('text.str.len() <20')

In [ ]:

df.info()

In [ ]:
df.describe()

In [ ]:
import seaborn as sns

sns.heatmap(df.corr(), annot=True)


In [ ]:
df['volunteers_count'].unique()

In [ ]:
df['judgment_confidence'].unique()

In [ ]:
class_counts = df['class'].value_counts()

class_counts

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt



sns.countplot(x='class', data=df)
plt.show()



### Text Preprocessing

In [ ]:

import re

def cleanHtml(sentence):
    cleanr = re.compile('<.*?>')
    cleantext = re.sub(cleanr, ' ', str(sentence))
    return cleantext


In [ ]:

def cleanPunc(sentence):
    cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', sentence)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = cleaned.strip()
    cleaned = cleaned.replace("\n"," ")
    return cleaned


In [ ]:


def removeSpace(sentence: str) -> str:
    sentence = sentence.strip()
    sentence = sentence.replace("\n"," ")
    return sentence



## Removing Stop Words

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
stop_words.update(['zero','one','two','three','four','five','six','seven','eight','nine','ten','may','also','across','among','beside','however','yet','within',])
re_stop_words = re.compile(r"\b(" + "|".join(stop_words) + ")\\W", re.I)

def removeStopWords(sentence):
    global re_stop_words
    return re_stop_words.sub(" ", sentence)



## removing Numbers

In [ ]:
def removeNumber(sentence):
    sentence = re.sub(r"\d+", "", sentence)
    return sentence


### Stemming

In [ ]:

from nltk.stem import SnowballStemmer

stemmer = SnowballStemmer("english")
def stemming(sentence):
    stemSentence = ""
    for word in sentence.split():
        stem = stemmer.stem(word)
        stemSentence += stem
        stemSentence += " "
    stemSentence = stemSentence.strip()
    return stemSentence

## removing url

In [ ]:
def removeUrl(sentence):
    cleaned = re.sub(r'http\S+', '', sentence)
    cleaned = cleaned.strip()
    cleaned = cleaned.replace("\n"," ")
    return cleaned



### Lemmatization



In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

lemmatizer = WordNetLemmatizer()

def lemmatizeWord(word):
    return lemmatizer.lemmatize(word)




## Feature Extraction

In [ ]:

def keepAlpha(sentence):
    alpha_sent = ""
    for word in sentence.split():
        alpha_word = re.sub('[^a-z A-Z]+', ' ', word)
        alpha_sent += alpha_word
        alpha_sent += " "
    alpha_sent = alpha_sent.strip()
    return alpha_sent

In [ ]:
import nltk

nltk.download('stopwords')

In [ ]:
from nltk.tokenize import TweetTokenizer
import nltk



def proccess_text(text):
    text = text.lower()
    text = removeUrl(text)
    text = removeNumber(text)
    text = removeStopWords(text)
    text = cleanHtml(text)
    text = cleanPunc(text)
    text = keepAlpha(text)
    text = removeSpace(text)

    ## stemming IS USELESS , used lemma instead 
    # text = stemming(text)

    
    

    
    words = TweetTokenizer(
        # strip_handles=True, reduce_len=True
    )
    
    words = [lemmatizeWord(word) for word in words.tokenize(text)]
    words = words.tokenize(text)
    return ' '.join(words)



df['text'] = df['text'].apply(proccess_text)

df['text']

In [ ]:
df['text'].iloc[4]

## Machine Learning Model 

### Proccessing  the Test data


In [ ]:
df_test = pd.read_csv('data/testing_data.csv')
df_test['text'] = df_test['text'].apply(proccess_text)
df_test


In [ ]:
df_test.iloc[0:10]

## for local testing

In [ ]:

from sklearn.model_selection import train_test_split
X_train =  df.drop('class',axis=1) 
y_train = df.drop(df.columns.difference(['class']), axis = 1)
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42)



X_train.shape, X_test.shape, y_train.shape, y_test.shape

## Vectorization 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(strip_accents='unicode', analyzer='word', ngram_range=(1, 3), norm='l2')


## Create our Pipeline

In [ ]:
from sklearn.pipeline import Pipeline

text_pipeline = Pipeline([
    ('tfidf', 
     vectorizer)
])


## Create Column Transformer  so that we can apply different preprocessing to different columns

In [ ]:
from sklearn.compose import ColumnTransformer


full_pipeline = ColumnTransformer([
    ('text', text_pipeline, 'text'),  
    (
        'volunteers_count', 
        'passthrough',
        ['volunteers_count']
    ),
    (
        'judgment_confidence', 
        'passthrough',
        ['judgment_confidence']
    )
])


## create  model pipleline (XGBoost)

In [ ]:
from xgboost import XGBClassifier


pipeline_with_xgb = Pipeline([
    ('preprocessing', full_pipeline),
    ('clf', XGBClassifier(
            tree_method='gpu_hist',
    ))      
])


## Applying GridSearchCV to find the best parameters (Fine Tuning)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'clf__learning_rate': [0,1,0.2],
    'clf__max_depth': [5],
    'clf__n_estimators': [200,400]
}

grid_search = GridSearchCV(pipeline_with_xgb, param_grid, cv=2 ,n_jobs=-1,
                            scoring='accuracy', verbose=2, return_train_score=True)


## training the model

In [ ]:

grid_search.fit(X_train, y_train)


## Predecting our test Data (in this case the local test data)

In [ ]:

print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

best_estimator = grid_search.best_estimator_
y_pred = best_estimator.predict(
    X_test
)

best_estimator


### Predecting Gloval Test Data

In [ ]:
X_test = df_test
y_pred = best_estimator.predict(
    X_test
)

## filling the submission file with the predictions

In [ ]:
import csv 
output_df = pd.DataFrame({'id': df_test['id'], 'class': y_pred})

output_df.to_csv('data/output.csv', index=False, quoting=csv.QUOTE_NONNUMERIC)

